<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
LangChain 커스텀 ChatModel 만들기
</div>

로컬 GPU에 올린 Hugging Face 모델을 **LangChain의 ChatModel**로 감싸서,
LangChain의 표준 인터페이스 3가지를 모두 사용해 봅니다.

| 기능 | 메서드 | 내부에서 호출되는 것 |
|---|---|---|
| 질의응답 (구 predict) | `llm.invoke(...)` | `_generate()` |
| 여러 질문 한 번에 | `llm.batch([...])` | `_generate()` × N |
| 생성되는 텍스트를 즉시 출력 | `llm.stream(...)` | `_stream()` ⭐ |

In [ ]:
%%capture
%pip install -q -U unsloth langchain-core

# GPU와 실행 환경 확인

Unsloth를 다른 Transformers 관련 라이브러리보다 먼저 가져오는 것이 좋습니다.

In [ ]:
from unsloth import FastLanguageModel
import torch

In [ ]:
print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 GPU: {torch.cuda.get_device_name(0)}")

# 4비트 Qwen2.5 모델 로딩

In [ ]:
# from unsloth import FastLanguageModel

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
#     max_seq_length=2048,
#     dtype=None,
#     load_in_4bit=True,
# )

# FastLanguageModel.for_inference(model)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

model.eval()

# 커스텀 ChatModel 클래스 작성

직접 구현할 핵심은 다음 네 가지입니다.

1. `model`, `tokenizer`를 Pydantic 필드로 선언
2. LangChain 메시지를 Qwen 채팅 형식으로 변환
3. `_generate()`에서 일반 응답 생성
4. `_stream()`에서 스트리밍 응답 생성


In [ ]:
import torch
from threading import Thread
from typing import Any, Iterator, List, Optional
from pydantic import ConfigDict

In [ ]:
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult

from transformers import TextIteratorStreamer

In [ ]:
class QwenChatModel(BaseChatModel):
    """Qwen2.5-Instruct를 LangChain ChatModel로 감싸는 클래스."""

    # BaseChatModel은 Pydantic 기반이므로 필드 선언이 필요합니다.
    model: Any
    tokenizer: Any

    max_tokens: int = 512
    do_sample: bool = True
    temperature: float = 0.7
    top_p: float = 0.9

    model_config = ConfigDict(arbitrary_types_allowed=True)

    @property
    def _llm_type(self) -> str:
        return "qwen2.5-custom-chatmodel"

    def _tokenize(self, messages: List[BaseMessage]):
        """LangChain 메시지를 Qwen 채팅 형식으로 변환하고 토큰화합니다."""
        chat = []

        for message in messages:
            if isinstance(message, SystemMessage):
                role = "system"
            elif isinstance(message, HumanMessage):
                role = "user"
            elif isinstance(message, AIMessage):
                role = "assistant"
            else:
                role = "user"

            chat.append({"role": role, "content": message.content})

        inputs = self.tokenizer.apply_chat_template(
            chat,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        return inputs.to(self.model.device)

    def _generation_options(self):
        """model.generate()에 공통으로 전달할 옵션입니다."""
        options = {
            "max_length": self.max_tokens,
            "do_sample": self.do_sample,
            "pad_token_id": self.tokenizer.pad_token_id,
        }

        if self.do_sample:
            options["temperature"] = self.temperature
            options["top_p"] = self.top_p

        return options

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        """invoke()와 batch()가 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)
        input_length = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                **self._generation_options(),
            )

        new_tokens = outputs[0][input_length:]
        text = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        return ChatResult(
            generations=[ChatGeneration(message=AIMessage(content=text))]
        )

    def _stream(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> Iterator[ChatGenerationChunk]:
        """stream()이 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)

        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        thread = Thread(
            target=self.model.generate,
            kwargs={
                **inputs,
                **self._generation_options(),
                "streamer": streamer,
            },
        )
        thread.start()

        for text in streamer:
            chunk = ChatGenerationChunk(
                message=AIMessageChunk(content=text)
            )

            if run_manager:
                run_manager.on_llm_new_token(text, chunk=chunk)

            yield chunk

        thread.join()

In [ ]:
llm = QwenChatModel(model=model, tokenizer=tokenizer)

# invoke — 기본 질의응답
- 문자열을 넣으면 → 자동으로 `HumanMessage`로 변환
- 메시지 리스트를 넣으면 → system 역할까지 지정 가능

In [ ]:
# 방법 1: 문자열로 간단히 질문
response = llm.invoke("LangChain이 무엇인지 두 문장으로 설명해 주세요.")
print(type(response).__name__)   # AIMessage
print(response.content)

In [ ]:
# 방법 2: 메시지 리스트로 system 역할 지정
messages = [
    SystemMessage(content="당신은 모든 답을 반말로 짧게 하는 친구입니다."),
    HumanMessage(content="파이썬 배우면 뭐가 좋아?"),
]
response = llm.invoke(messages)
print(response.content)

# batch — 여러 질문 한 번에 처리

`batch()`는 리스트의 각 항목에 대해 내부적으로 `_generate()`를 호출합니다.

> ⚠️ **로컬 GPU 모델 주의점**: LangChain의 `batch()`는 기본적으로 **여러 스레드로 동시 실행**을
> 시도합니다. API 서버라면 빨라지지만, GPU가 1개뿐인 로컬 모델은 동시 호출 시 충돌/속도 저하가
> 생길 수 있습니다. → `max_concurrency=1`로 **순차 실행**을 지정하는 것이 안전합니다.


In [ ]:
questions = [
    "지구에서 가장 큰 바다는?",
    "1 + 1 은 왜 2인가요? 한 문장으로.",
    "김밥의 주재료 3가지만 알려주세요.",
]

# 로컬 GPU 1개 → 순차 실행이 안전
answers = llm.batch(questions, config={"max_concurrency": 1})

for q, a in zip(questions, answers):
    print(f"Q: {q}")
    print(f"A: {a.content}")
    print("-" * 60)


# stream — 생성되는 텍스트를 실시간으로 출력

`model.generate()`는 응답 생성이 끝날 때까지 기다리는 함수입니다.
따라서 백그라운드 스레드에서 모델을 실행하고,
메인 스레드에서는 `TextIteratorStreamer`가 전달하는 텍스트를 출력합니다.

Jupyter에서 바로 표시되도록 `flush=True`를 사용합니다.

In [ ]:
for chunk in llm.stream("가을에 대한 시를 한 편 지어주세요."):
    print(chunk.content, end="", flush=True)

In [ ]:
for chunk in llm.stream("gitea vs gitlab 차이점을 설명해 주세요."):
    print(chunk.content, end="", flush=True)